# OGLE data access: from EWS page to normalized observations

This notebook is an executable inspection of the public OGLE-IV Early Warning System adapter. It shows the raw web table, the normalized target table, optional sky/magnitude filters, and the per-event `phot.dat` observations.

In [5]:
from io import StringIO
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import requests
from target_selection import data_source


SEASON = 2026
CACHE_DIR = Path('outputs/cache/ogle_photometry')
EWS = data_source.ogle
EWS_URL = EWS.describe()['catalog_url']
EWS.describe(), EWS_URL

({'name': 'ogle',
  'kind': 'survey',
  'features': ['targets', 'parameters', 'epochs', 'photometry']},
 'https://ogle.astrouw.edu.pl/ogle4/ews/ews.html')

## 1. Inspect the public EWS page

The adapter downloads the HTML page, decodes it as Latin-1, and reads its event table with `pandas.read_html`. This cell shows the raw response and the columns before normalization.

In [8]:
raw_tables

[     Unnamed: 0          Event      Field  Star No   RA (J2000)  Dec (J2000)  \
 0           NaN  2026-BLG-0001  BLG499.21     4189  17:19:55.45  -30:16:43.4   
 1           NaN  2026-BLG-0002  BLG500.01    92092  17:54:08.86  -29:01:25.1   
 2           NaN  2026-BLG-0003  BLG500.04    30440  17:52:14.90  -29:00:12.6   
 3           NaN  2026-BLG-0004  BLG500.05   113286  17:50:57.48  -29:01:04.8   
 4           NaN  2026-BLG-0005  BLG500.19    31825  17:53:36.31  -28:24:33.7   
 ..          ...            ...        ...      ...          ...          ...   
 932         NaN  2026-BLG-0910  BLG657.04   101532  18:19:42.32  -30:55:20.0   
 933         NaN  2026-BLG-0911  BLG661.06    36545  17:38:23.60  -34:02:47.8   
 934         NaN  2026-BLG-0912  BLG909.07     3828  17:07:39.29  -34:36:51.0   
 935         NaN  2026-BLG-0913  BLG932.06    13817  17:29:56.76  -40:48:05.3   
 936         NaN  2026-BLG-0914  BLG994.20    18692  17:07:05.19  -37:38:45.7   
 
       Tmax (HJD)      Tma

In [2]:
response = requests.get(EWS_URL, timeout=60)
response.raise_for_status()
raw_html = response.content.decode('latin-1')
raw_tables = pd.read_html(StringIO(raw_html))
raw_events = raw_tables[0]
print(f'HTTP status: {response.status_code}')
print(f'HTML bytes: {len(response.content):,}')
print(f'Tables found: {len(raw_tables)}')
print('Raw columns:')
display(raw_events.head())

HTTP status: 200
HTML bytes: 278,055
Tables found: 1
Raw columns:


,Unnamed: 0,Event,Field,Star No,RA (J2000),Dec (J2000),Tmax (HJD),Tmax (UT),tau,Umin,Amax,Dmag,fbl,Ibl,I0
0,NaN,2026-BLG-0001,BLG499.21,4189,17:19:55.45,-30:16:43.4,2461065.134,2026-01-24.63,42.838,0.183,5.534,1.858,1.000,17.971,17.971
1,NaN,2026-BLG-0002,BLG500.01,92092,17:54:08.86,-29:01:25.1,2461093.940,2026-02-22.44,453.842,0.014,72.729,2.246,0.096,19.433,21.974
2,NaN,2026-BLG-0003,BLG500.04,30440,17:52:14.90,-29:00:12.6,2461183.814,2026-05-23.31,157.964,0.563,1.980,0.742,1.000,17.718,17.718
3,NaN,2026-BLG-0004,BLG500.05,113286,17:50:57.48,-29:01:04.8,2461090.285,2026-02-18.79,29.318,0.677,1.719,0.588,1.000,17.646,17.646
4,NaN,2026-BLG-0005,BLG500.19,31825,17:53:36.31,-28:24:33.7,2461105.413,2026-03-05.91,24.755,0.266,3.856,0.912,0.461,17.584,18.424


## 2. Load normalized OGLE targets

The adapter converts sexagesimal coordinates to degrees and maps the preliminary microlensing model parameters `Tmax`, `tau` and `Umin` to `t0_hjd`, `tE_days` and `u0`. `Tmax` is the fitted time of maximum magnification, `tau` is the fitted event timescale, and `Umin` is the minimum lens-source separation in Einstein-radius units. These are preliminary EWS parameters and can change as the event develops. The original OGLE columns and event URLs remain available.

In [3]:
events = EWS.load_targets(season=SEASON)
print(f'Loaded targets: {len(events):,}')
display(events[['target_id', 'ra_deg', 'dec_deg', 't0_hjd', 'tE_days', 'u0', 'category', 'photometry_url']].head())

Loaded targets: 937


,target_id,ra_deg,dec_deg,t0_hjd,tE_days,u0,category,photometry_url
0,2026-BLG-0001,259.981042,-30.278722,2461065.134,42.838,0.183,ews_candidate,https://www.astrouw.edu.pl/ogle/ogle4/ews/2026...
1,2026-BLG-0002,268.536917,-29.023639,2461093.940,453.842,0.014,ews_candidate,https://www.astrouw.edu.pl/ogle/ogle4/ews/2026...
2,2026-BLG-0003,268.062083,-29.003500,2461183.814,157.964,0.563,ews_candidate,https://www.astrouw.edu.pl/ogle/ogle4/ews/2026...
3,2026-BLG-0004,267.739500,-29.018000,2461090.285,29.318,0.677,ews_candidate,https://www.astrouw.edu.pl/ogle/ogle4/ews/2026...
4,2026-BLG-0005,268.401292,-28.409361,2461105.413,24.755,0.266,ews_candidate,https://www.astrouw.edu.pl/ogle/ogle4/ews/2026...


## 3. Filter by magnitude and sky region

`magnitude_range` uses the OGLE EWS I-band baseline magnitude (`Ibl`). `ra_range` and `dec_range` use degrees. A right-ascension interval that crosses 0 degrees is supported.

In [ ]:
filtered_events = EWS.load_targets(
    season=SEASON,
    magnitude_range=(None, 17.0),
    ra_range=(250.0, 280.0),
    dec_range=(-35.0, -20.0),
)
print(f'Filtered targets: {len(filtered_events):,}')
display(filtered_events[['target_id', 'ra_deg', 'dec_deg', 'magnitude_baseline_I']].head())

The same magnitude filter can be expressed through the generic provider filter interface:

In [ ]:
bright_events = EWS.load_targets(
    season=SEASON,
    filters={'magnitude': {'maximum': 17.0}},
)
print(f'OGLE EWS targets with Ibl <= 17: {len(bright_events):,}')

## 4. Download and inspect one event light curve

Each event page links to a public `phot.dat` file. The adapter caches it locally and returns one normalized observation row per epoch.

In [ ]:
sample = filtered_events.head(1) if len(filtered_events) else events.head(1)
observations = EWS.photometry(sample, cache_dir=CACHE_DIR)
print(f'Observation rows: {len(observations):,}')
display(observations.head())

if len(observations):
    plt.figure(figsize=(9, 4))
    plt.errorbar(observations['epoch_hjd'], observations['magnitude'], yerr=observations['magnitude_err'], fmt='.', ms=3)
    plt.gca().invert_yaxis()
    plt.xlabel('HJD')
    plt.ylabel('OGLE I magnitude')
    plt.title(f'OGLE EWS photometry: {observations["target_id"].iloc[0]}')
    plt.grid(alpha=0.25)
    plt.show()

## 5. Inspect provenance and cache

The target table keeps the event URL, photometry URL and archive URL. The downloaded text files are stored in the selected cache directory and can be reused in later runs.

In [ ]:
display(sample[['target_id', 'event_url', 'photometry_url', 'archive_url', 'finding_chart_url']])
print(f"Cached files: {len(list(CACHE_DIR.glob('*.phot.dat')))} in {CACHE_DIR}")